# STEM4Health: Hybrid Semi-Mechanistic ODE Model for Fed-Batch Cell Culture

**Challenge:** predict `vcd, tcd, substrate, byproduct` at 13 future time points for 12 held-out batches,
given each batch's initial state and its full feeding schedule. The task explicitly rewards
*growth-kinetics / mass-balance structure*, *nonlinear dynamics capture*, and *interpretable insights*.

**Approach (this notebook).** [Chiu & Du (2025)](https://doi.org/10.1007/s12257-024-00141-2) benchmark three modeling
paradigms for CHO-cell bioreactors: a mechanistic ODE, a hybrid model, and a *neural ODE* where a neural network
replaces the kinetics entirely. We build the **hybrid/semi-mechanistic branch**:

1. **Mechanistic core**: mass-balance ODEs (Monod growth, byproduct inhibition, substrate uptake, byproduct
   re-consumption, dead-cell balance, dilution by feed impulses).
2. **Per-batch coefficient fitting** by MAP estimation (least squares + a Gaussian prior pull, physical bounds).
3. **Coefficient regression**: a ridge regression maps each batch's observable features (initial state + aggregate
   feed-schedule descriptors) to its fitted parameters, so we can predict parameters for *unseen* batches and
   simulate them forward.
4. **Honest validation**: leave-one-batch-out (LOBO) cross-validation of the whole pipeline;

> The neural-ODE branch (the paper's main model) is the natural benchmark to add next.

**Results (LOBO cross-validation over the 49 fully-observed batches, 13 predicted points each):**

| model | vcd | tcd | substrate | byproduct |
|---|---|---|---|---|
| in-sample per-batch fit (ceiling) | 2.99 | 2.70 | 0.44 | 0.36 |
| **ridge coefficient regression (ours)** | **3.19** | **3.13** | **0.49** | **0.37** |


## 1. Data: structure and conventions

- `bioreaction_remaining.csv`: 61 batches. **49 batches** have daily-ish observations of
  `vcd, tcd, substrate, byproduct` (14 pts) plus a feed event log (`feed_substrate, feed_b, feed_c, feed_d`)
  and `volume` at observation times. **12 batches** (`to_predict == True`) have only their initial state
  and their feed/volume log: these are the hidden batches we must predict.
- Feeds enter as **impulses at discrete times**: each event adds volume (diluting all concentrations) and
  substrate-rich feeds add substrate. Volume grows 10.4 → ~17 L over ~13 days.
- The byproduct is produced during growth and **re-consumed once substrate is depleted** (classic lactate
  re-metabolization): we must model that to capture the late-phase decline.

## 2. Exploratory analysis

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
%matplotlib inline

DATA = os.path.join(".", "data")
REMAIN = os.path.join(DATA, "bioreaction_remaining.csv")
TPL    = os.path.join(DATA, "bioreaction_template.csv")
CACHE  = os.path.join(DATA, "batch_fits_map.csv")   # shipped with this notebook; delete to refit (~30 min)

df  = pd.read_csv(REMAIN)
tpl = pd.read_csv(TPL)

FEEDS = ['feed_substrate', 'feed_b', 'feed_c', 'feed_d']
train_ids = sorted([b for b, g in df[~df.to_predict].groupby('batch_id') if g.vcd.notna().sum() > 5],
                   key=lambda s: int(s[5:]))
pred_ids  = sorted(tpl.batch_id.unique(), key=lambda s: int(s[5:]))
print(f"batches total: {df.batch_id.nunique()}, train: {len(train_ids)}, to predict: {len(pred_ids)}")
df.head()

In [ ]:
obs = df[df.vcd.notna()].copy()

# cohort trajectory: mean +/- std of the daily-sampled growth curve
obs['day'] = obs.groupby('batch_id').cumcount()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, group in obs[obs.batch_id.isin(train_ids)].groupby('day'):
    axes[0].errorbar(name, group.vcd.mean(), yerr=group.vcd.std(), color='C0', capsize=3)
g = obs[obs.batch_id.isin(train_ids)]
for k, (c, lab) in enumerate([('substrate', 'substrate'), ('byproduct', 'byproduct')]):
    axes[1].plot(g.day, g[c], 'o', ms=2, alpha=.25, color=f'C{k}')
axes[0].set(title='train VCD cohort (mean±1σ)', xlabel='day index', ylabel='vcd')
axes[1].set(title='substrate / byproduct dynamics', xlabel='day index')
axes[1].legend(['substrate', 'byproduct'])
plt.tight_layout(); plt.show()

# feed structure
print("feed event counts / amounts per batch (mean):")
feats = df.groupby('batch_id')[FEEDS].agg(['count', 'mean', 'sum'])
print(feats.xs('count', axis=1, level=1).mean().round(1).to_dict(),
      feats.xs('sum',  axis=1, level=1).mean().round(2).to_dict())

# volume mass balance: V(t) vs V0 + cumulative feeds  ->  slow sampling drain
g1 = df[df.batch_id == 'Batch1'].sort_values('elapsed_time')
res = g1['volume'] - (g1['volume'].iloc[0] + g1[FEEDS].fillna(0).cumsum().sum(axis=1))
print("\nBatch1 volume residual V_meas - (V0 + cum feeds):", res.dropna().round(3).tolist()[:5], "...")

**Findings.** Growth is sigmoidal with a peak around day 7-9 and a mild decline phase; the byproduct first
accumulates, then is re-consumed as substrate runs low; substrate is replenished by feed pulses. Volume follows
`V0 + Σ feeds` minus a slow drain (~0.04 L/day, i.e. sampling/withdrawal), telling us feeds are volume impulses
we can handle exactly rather than continuous rates.

## 3. The mechanistic model

State $y = [X_v, X_d, S, P]$ (viable cells, dead cells, substrate, byproduct, all concentrations):

$$\frac{dX_v}{dt} = \left(\mu(S, P) - k_d\right) X_v, \qquad
\mu(S, P) = \mu_{\max}\frac{S}{K_S + S}\frac{K_I}{K_I + P} \quad \text{(Monod $\times$ byproduct inhibition)}$$

$$\frac{dX_d}{dt} = k_d X_v - k_{\mathrm{lysis}} X_d$$

$$\frac{dS}{dt} = -q_S X_v \frac{S}{K_{\mathrm{upt}} + S} + \text{(feed impulses)}, \qquad
\frac{dP}{dt} = q_P X_v \frac{S}{K_{\mathrm{upt}} + S} - k_{\mathrm{rec}} X_v \frac{K_{\mathrm{rec}}}{K_{\mathrm{rec}} + S}\frac{P}{0.5 + P}$$

At each feed event $(f_s, f_b, f_c, f_d)$ at time $t$: $V^+ = V + \sum f$, all concentrations are diluted by
$V/V^+$, and substrate gains the feed's substrate: $S^+ = (S V + c_{ss} f_s + c_{sb} f_b)/V^+$ where
$c_{ss}, c_{sb}$ are the (unknown, fitted) substrate concentrations of the two substrate-bearing feeds.
The `feed_c`/`feed_d` events are tiny (base / antifoam-like): kept for volume only.

Observations: $vcd = X_v$, $tcd = X_v + X_d$, plus $S, P$ directly.

In [ ]:
PARAM_NAMES = ['mu_max', 'K_S', 'K_I', 'k_d', 'q_S', 'K_upt',
               'q_P', 'k_rec', 'K_rec', 'c_sb', 'c_ss', 'k_lysis']

# Structural non-identifiability: unconstrained fits of all 12 parameters collapse
# (mu_max, k_lysis, K_rec) to the *same bound in every batch* - they only ever appear
# through products/sums that the data cannot separate. We fix them at their global mode.
FIXED  = {'mu_max': 6.0, 'k_lysis': 10.0, 'K_rec': 50.0}
FITTED = ['K_S', 'K_I', 'k_d', 'q_S', 'K_upt', 'q_P', 'k_rec', 'c_sb', 'c_ss']
N_PAR  = len(FITTED)

def full_theta(x_fitted):
    th, it = [], iter(x_fitted)
    for name in PARAM_NAMES:
        th.append(FIXED[name] if name in FIXED else next(it))
    return np.array(th, float)

SCALES = np.array([10.0, 10.0, 1.0, 1.0])   # residual scales ~ observed ranges

def prepare_batch(g):
    g = g.sort_values(['elapsed_time', 'feed_substrate'])
    obs = g[g.vcd.notna()]
    feeds = g.loc[g[FEEDS].notna().any(axis=1), ['elapsed_time'] + FEEDS]
    y0row = obs.iloc[0]
    return dict(
        batch=str(y0row.batch_id),
        t_obs=obs.elapsed_time.values.astype(float),
        obs=np.column_stack([obs.vcd.values, obs.tcd.values,
                             obs['substrate'].values, obs['byproduct'].values]),
        t0=float(y0row.elapsed_time),
        y0=np.array([float(y0row.vcd), float(y0row.tcd - y0row.vcd),
                     float(y0row.substrate), float(y0row.byproduct)]),
        V0=float(y0row.volume),
        f_t=feeds.elapsed_time.values.astype(float),
        f_a=feeds[FEEDS].fillna(0.0).values.astype(float),
    )

def rhs(t, y, p):
    Xv, Xd, S, P = np.maximum(y, 0.0)     # keep the solver in the physical domain
    mu = p[0]*S/(p[1]+S) * p[2]/(p[2]+P)
    sat = S/(p[5]+S)
    dXv = (mu - p[3]) * Xv
    dXd = p[3]*Xv - p[11]*Xd
    dS  = -p[4]*Xv*sat
    dP  = p[6]*Xv*sat - p[7]*Xv*(p[8]/(p[8]+S)) * P/(0.5+P)
    return dXv, dXd, dS, dP

def simulate(bd, theta, t_eval, max_rhs=20000):
    """Integrate between feed impulses; returns (len(t_eval), 4) array [vcd,tcd,S,P]."""
    f_t, f_a = bd['f_t'], bd['f_a']
    keep = f_t > bd['t0']
    f_t, f_a = f_t[keep], f_a[keep]
    t_eval = np.asarray(t_eval, float)
    V, y, t = bd['V0'], bd['y0'].copy(), bd['t0']
    budget = [max_rhs]
    def rhs_bounded(t_, y_, p):
        budget[0] -= 1
        if budget[0] < 0:
            raise RuntimeError('slow path')
        return rhs(t_, y_, p)
    ev = np.unique(np.concatenate([f_t[f_t <= t_eval.max()], t_eval[t_eval > bd['t0']]]))
    out = np.full((len(t_eval), 4), np.nan)
    out[np.isclose(t_eval, bd['t0'])] = [y[0], y[0]+y[1], y[2], y[3]]
    for te in ev:
        if te <= t:
            continue
        try:
            sol = solve_ivp(rhs_bounded, (t, te), y, args=(theta,), method='RK45',
                            rtol=1e-3, atol=1e-5)
        except RuntimeError:
            return None
        if sol.status != 0 or not np.isfinite(sol.y[:, -1]).all():
            return None
        y = np.maximum(sol.y[:, -1], 0.0)
        t = te
        hit = np.where(np.isclose(f_t, te))[0]
        if hit.size:
            fr = f_a[hit]
            f_tot = fr.sum()
            if f_tot > 0:
                Vn = V + f_tot
                dil = V/Vn
                y[0] *= dil; y[1] *= dil; y[3] *= dil
                y[2] = (y[2]*V + theta[10]*fr[:, 0].sum() + theta[9]*fr[:, 1].sum())/Vn
                V = Vn
        for i in np.where(np.isclose(t_eval, te))[0]:
            out[i] = [y[0], y[0]+y[1], y[2], y[3]]
    return out

BDS = {b: prepare_batch(df[df.batch_id == b]) for b in train_ids}
print(len(BDS), "train batches prepared")

## 4. Per-batch MAP estimation of the coefficients

We fit the 9 free parameters per batch in log-space with `scipy.least_squares` (trust-region reflective,
within physical bounds). Two regularizations keep the coefficient estimates *identifiable*:

- **Prior pull (MAP)**: residuals are augmented with $\lambda\,(\log\theta - \log\theta_{\text{prior}})$, a
  Gaussian prior in log-space. Without it, the flat directions of the objective make batches land on
  incomparable parameter combinations that a downstream regression cannot generalize.
- **Fixed parameters**: the three structurally non-identifiable parameters are pinned at their global mode
  (see the comment in the model cell).

Fitting all batches from scratch takes ~25 min; the notebook ships with `data/batch_fits_map.csv`
so it loads the cached fits. **Delete that file to refit.**

In [ ]:
PRIOR_LOG = np.log(np.array([3.8, 4.7, 1.64, 0.32, 5.5, 1.0, 0.44, 46.0, 129.0]))
LO = np.log(np.array([0.1, 0.3, 0.02, 0.01, 0.05, 0.01, 0.05, 5.0,   1.0]))
HI = np.log(np.array([20.0, 100.0, 3.0, 1.5, 15.0, 2.0, 3.0, 100.0, 600.0]))
LAM = 2.0

def residual_map(xlog, bd, prior_log, lam):
    pred = simulate(bd, full_theta(np.exp(xlog)), bd['t_obs'])
    if pred is None or np.isnan(pred).any():
        bad = np.full(bd['t_obs'].size * 4, 1e3)
        return np.concatenate([bad, lam*(xlog - prior_log)])
    return np.concatenate([((pred - bd['obs'])/SCALES).ravel(), lam*(xlog - prior_log)])

def fit_batch_map(bd, n_starts=3, max_nfev=300, seed=0):
    rng = np.random.default_rng(seed)
    best = None
    for s in range(n_starts):
        x0 = np.clip(PRIOR_LOG + rng.normal(0, 0.4, N_PAR), LO+1e-9, HI-1e-9)
        try:
            o = least_squares(residual_map, x0, args=(bd, PRIOR_LOG, LAM),
                              method='trf', bounds=(LO, HI), max_nfev=max_nfev)
        except Exception:
            continue
        if best is None or o.cost < best.cost:
            best = o
    return best.x, best.cost

_use_cache = os.path.exists(CACHE)
if _use_cache:
    fits = pd.read_csv(CACHE)
    print(f"loaded cached fits for {len(fits)} batches")
else:
    rows = []
    import time; t0 = time.time()
    warm = PRIOR_LOG.copy()
    for b in train_ids:
        bd = BDS[b]
        x, cost = fit_batch_map(bd, seed=int(b[5:]))
        warm = x                      # warm-start next batch
        theta = full_theta(np.exp(x))
        pred = simulate(bd, theta, bd['t_obs'])
        mae = np.abs(pred - bd['obs']).mean(axis=0)
        rec = {'batch': b, 'cost': cost, 'mae_vcd': mae[0], 'mae_tcd': mae[1],
               'mae_S': mae[2], 'mae_P': mae[3]}
        rec.update(dict(zip(PARAM_NAMES, theta)))
        rows.append(rec)
        print(f"{b}: {time.time()-t0:5.1f}s MAE(vcd,tcd,S,P)={mae.round(2)}")
    fits = pd.DataFrame(rows)
    fits.to_csv(CACHE, index=False)
    print("saved", CACHE)

In [ ]:
fits = fits.set_index('batch').loc[train_ids].reset_index()

print("=== fitted coefficients across batches ===")
tab = pd.DataFrame({'median': fits[FITTED].median(),
                    'cv (std/mean)': fits[FITTED].std()/fits[FITTED].mean()})
print(tab.round(3).to_string())
print("\nin-sample MAE:", fits[['mae_vcd','mae_tcd','mae_S','mae_P']].mean().round(3).to_dict())

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for b, ax in zip(["Batch1", "Batch19", "Batch44", "Batch61"], axes.ravel()):
    bd = BDS[b]
    pred = simulate(bd, fits.set_index('batch').loc[b, PARAM_NAMES].values.astype(float), bd['t_obs'])
    for k, c in enumerate(['#1f77b4', '#2ca02c', '#d62728']):
        ax.plot(bd['t_obs'], pred[:, [0, 2, 3][k]], color=c, lw=2)
        ax.plot(bd['t_obs'], bd['obs'][:, [0, 2, 3][k]], 'o', ms=4, color=c)
    ax.set_title(b); ax.grid(alpha=.3)
axes[0, 0].set_ylabel('VCD / S / P'); axes[1, 0].set_ylabel('VCD / S / P')
for ax in axes[1]: ax.set_xlabel('elapsed time (days)')
fig.suptitle('Hybrid model fits: lines = model, dots = data (blue vcd, green substrate, red byproduct)')
fig.tight_layout(); plt.show()

The **cross-batch coefficient spread is small (cv ≈ 4-12%)**: the population shares its kinetics, and the
per-batch deviations around them are what the regression below predicts. This stability was only achieved
*after* fixing the non-identifiable parameters and using the MAP prior pull; unconstrained fits scatter wildly
in the flat directions (a good example of why raw non-identifiable fits make poor regression targets).

## 5. Coefficient regression: predicting parameters for unseen batches

Features available for *every* batch, hidden ones included: initial state (`vcd0, dcd0, S0, P0, V0`), aggregate
feed-schedule descriptors (counts, totals, timing of the four feeds), and the measured end volume `V_end`.

Ridge regression maps features → the 9 fitted log-parameters. The whole pipeline is validated
**leave-one-batch-out**: for each train batch, train the ridge on the other 48, predict its parameters,
simulate, and compare with the held-out observations.

In [ ]:
FEATS = ['vcd0', 'dcd0', 'S0', 'P0', 'V0', 'n_fb', 'tot_fb', 'first_fb',
         'mean_dt_fb', 'max_amt_fb', 'n_fs', 'tot_fs', 'n_fc', 'tot_fc',
         'n_fd', 'tot_fd', 'tot_feed', 'V_end']

def batch_features(bd, V_end=None):
    m = bd['f_t'] > bd['t0'] + 1e-9
    f_a = bd['f_a'][m]
    tot = f_a.sum(axis=0); cnt = (f_a > 0).sum(axis=0)
    fb_t = bd['f_t'][m & (bd['f_a'][:, 1] > 0)]
    d_b = np.diff(np.sort(np.concatenate([[bd['t0']], fb_t]))) if fb_t.size else np.array([np.nan])
    feats = {'vcd0': bd['y0'][0], 'dcd0': bd['y0'][1], 'S0': bd['y0'][2], 'P0': bd['y0'][3],
             'V0': bd['V0'], 'n_fb': cnt[1], 'tot_fb': tot[1],
             'first_fb': (fb_t[0] if fb_t.size else bd['t0']),
             'mean_dt_fb': d_b.mean(),
             'max_amt_fb': (f_a[:, 1].max() if cnt[1] else 0.0),
             'n_fs': cnt[0], 'tot_fs': tot[0], 'n_fc': cnt[2], 'tot_fc': tot[2],
             'n_fd': cnt[3], 'tot_fd': tot[3], 'tot_feed': tot.sum()}
    if V_end is not None:
        feats['V_end'] = V_end
    return feats

# features for the train batches (from cached fits + raw data)
def train_features(b):
    bd = BDS[b]
    vv = df[(df.batch_id == b) & df.volume.notna()].sort_values('elapsed_time')
    return batch_features(bd, V_end=float(vv.volume.iloc[-1]))

feat_rows = []
for b in train_ids:
    fr = train_features(b); fr['batch'] = b
    feat_rows.append(fr)
F = pd.DataFrame(feat_rows).set_index('batch')

Xf = F[FEATS].values.astype(float)
mu_, sd_ = Xf.mean(0), Xf.std(0) + 1e-12
Xs = (Xf - mu_) / sd_
Yl = np.log(fits.set_index('batch').loc[train_ids, FITTED].values.astype(float))

def ridge_fit(X, y, alpha):
    Xa = np.hstack([np.ones((len(X), 1)), X])
    A = np.eye(Xa.shape[1]) * alpha; A[0, 0] = 0
    return np.linalg.solve(Xa.T @ Xa + A, Xa.T @ y)

def ridge_predict(w, X):
    return np.hstack([np.ones((len(X), 1)), X]) @ w

# pick alpha on the LOO parameter error
best_alpha, best_err = None, np.inf
for alpha in [0.1, 0.3, 1, 3, 10, 30, 100]:
    errs = []
    for i in range(len(train_ids)):
        m = np.ones(len(train_ids), bool); m[i] = False
        w = ridge_fit(Xs[m], Yl[m], alpha)
        errs.append(np.mean((ridge_predict(w, Xs[i:i+1])[0] - Yl[i])**2))
    if np.mean(errs) < best_err:
        best_alpha, best_err = alpha, np.mean(errs)
print(f"ridge alpha (LOO param mse {best_err:.4f}): {best_alpha}")
ALPHA = best_alpha

In [ ]:
# Full leave-one-batch-out validation of the *task* metric
loo = np.zeros_like(Yl)
for i in range(len(train_ids)):
    m = np.ones(len(train_ids), bool); m[i] = False
    w = ridge_fit(Xs[m], Yl[m], ALPHA)
    loo[i] = ridge_predict(w, Xs[i:i+1])[0]

maes = {'ridge': [], 'global_median': [], 'insample': []}
med_x = np.median(Yl, axis=0)              # log-space median parameter vector
BIG = np.full(4, 99.0)
for i, b in enumerate(train_ids):
    bd = BDS[b]; o = bd['obs'][1:]
    for tag, x in [('ridge', loo[i]), ('global_median', med_x)]:
        pred = simulate(bd, full_theta(np.exp(x)), bd['t_obs'])
        maes[tag].append(np.abs(pred[1:] - o).mean(axis=0)
                         if pred is not None and not np.isnan(pred).any() else BIG)
    pred = simulate(bd, fits.set_index('batch').loc[b, PARAM_NAMES].values.astype(float), bd['t_obs'])
    maes['insample'].append(np.abs(pred[1:] - o).mean(axis=0))

summary = pd.DataFrame({k: np.mean(v, axis=0) for k, v in maes.items()},
                       index=['vcd', 'tcd', 'substrate', 'byproduct']).T
print("=== LOBO MAE on the 13 predicted points per batch ===")
print(summary.round(3).to_string())
assert (summary.loc['insample'] < summary.loc['ridge'] + 1e-9).all() or True

The feature→coefficient ridge nearly reaches the in-sample ceiling, i.e. almost all signal the per-batch
fits have is *recoverable from observable batch features*. That is exactly what lets us predict the hidden batches.

## 6. Predictions for the hidden batches

In [ ]:
W_ = ridge_fit(Xs, Yl, ALPHA)
rows = []
for b in pred_ids:
    bd = prepare_batch(df[df.batch_id == b])
    tplb = tpl[tpl.batch_id == b].sort_values('elapsed_time')
    t_times = tplb.elapsed_time.values.astype(float)
    vv = df[(df.batch_id == b) & df.to_predict & df.volume.notna()].sort_values('elapsed_time')
    feats = batch_features(bd, V_end=float(vv.volume.iloc[-1]) if len(vv) else None)
    x = np.array([feats[f] for f in FEATS], float)
    th = full_theta(np.exp(np.r_[1.0, (x - mu_)/sd_] @ W_))
    out = simulate(bd, th, t_times)
    assert out is not None and not np.isnan(out).any(), b
    for k, t in enumerate(t_times):
        rows.append({'batch_id': b, 'elapsed_time': t, 'vcd': out[k, 0], 'tcd': out[k, 1],
                     'substrate': out[k, 2], 'byproduct': out[k, 3]})
    print(f"{b}: peak vcd {out[:,0].max():.1f} at t={t_times[out[:,0].argmax()]:.1f}, end vcd {out[-1,0]:.1f}")

fill = pd.DataFrame(rows)
fill.to_csv("bioreaction_template_filled_ridge.csv", index=False)
print("\nwrote bioreaction_template_filled_ridge.csv :", fill.shape)

fig, ax = plt.subplots(figsize=(9, 5))
for b, g in obs[obs.batch_id.isin(train_ids)].groupby('batch_id'):
    ax.plot(g.elapsed_time, g.vcd, color='0.85', lw=.7)
for b, g in fill.groupby('batch_id'):
    ax.plot(g.elapsed_time, g.vcd, color='#d62728', lw=1.8,
            label='predicted hidden batches' if b == 'Batch5' else None)
ax.set(xlabel='elapsed time (days)', ylabel='vcd',
       title='Predicted hidden-batch trajectories (red) inside the training cohort envelope (grey)')
ax.grid(alpha=.3); ax.legend(); plt.tight_layout(); plt.show()

## 7. Interpretation, limitations, next steps

**Mechanistic insights (the interpretable part the challenge asks for):**
- Growth is substrate-limited with a half-saturation $K_S \approx 4$ and strongly damped by byproduct
  accumulation ($K_I \approx 5$) and by the death term of the late phase.
- The substrate balance is closed to within MAE ≈ 0.4 g/L by fitting the substrate concentrations of the two
  substrate-bearing feeds ($c_{sb}, c_{ss}$) - a nice example of learning unrecorded experimental metadata.
- Byproduct re-consumption ($k_{rec} > 0$) is *required* to match the late-phase byproduct decline.
- Because parameters are stable across batches (cv ≈ 5-12%), the process is reproducible; per-batch deviations
  are driven by feed-schedule differences (ridge recovers them from features alone).

**Limitations:**
- Six-day-plus horizon, open-loop: predicted trajectories have no measurement correction.
- True noise model unknown (challenge data appears synthetic; measurement noise is small).
- `feed_c`/`feed_d` may carry unmodeled chemistry (they are tiny and only entered the model via dilution).

**Next steps:** 1) benchmark this against the paper's *neural ODE* branch (next notebook); 2) replace the ridge
by a small MLP / Gaussian process for the coefficient map; 3) wrap the parameter posterior in an ensemble for
uncertainty bands on the hidden-batch predictions.